# Does Longer Prompt Padding Destabilize LLM Math Answers?

This notebook demonstrates the analysis pipeline from `method.py`: given repeated LLM samples on length-and-content-matched GSM8K prompts (a bare-question control, plus relevant-elaboration and irrelevant-filler padding at short/medium/long token-count tiers), it computes per-`(prompt, model)` **answer variance / coefficient of variation (CV)**, **accuracy**, and a **logprob-derived entropy proxy**, then builds the built-in baseline comparison (bare control vs. filler vs. relevant content at each length tier).

**Demo data**: a curated subset of the real run's logged completions (2 GSM8K seeds x all 7 prompt variants x 3 models x 5 samples each = 210 raw completion rows), loaded from `mini_demo_data.json`. This lets the notebook demonstrate the exact same aggregation/statistics code (`aggregate_results`, `build_summary_stats`, `build_baseline_comparison`) that ran on the full 6,720-call dataset, without re-issuing paid OpenRouter API calls. The original prompt-generation code (`build_dataset.py`) and the async OpenRouter-sampling code (`call_openrouter` / `sample_one` / `run_all`) are reproduced below unmodified for reference; the demo drives the pipeline from the pre-collected raw completions instead of live API calls.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# aiohttp, loguru, tenacity, tiktoken, datasets -- NOT pre-installed on Colab, always install
_pip('aiohttp==3.11.11')
_pip('loguru==0.7.3')
_pip('tenacity==9.0.0')
_pip('tiktoken==0.8.0')

# numpy, pandas, scipy, matplotlib -- pre-installed on Colab, install locally only (match Colab versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import asyncio
import json
import math
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path

import aiohttp
import numpy as np
import pandas as pd
from loguru import logger
from scipy.stats import entropy as scipy_entropy
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

import matplotlib.pyplot as plt  # for the results visualization cell

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-0e9809-interpretive-load-not-token-count-drives/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
prompts = data["prompts"]
raw_completions = data["raw_completions"]
print(f"Loaded {len(prompts)} matched prompts (from {len({p['seed_id'] for p in prompts})} GSM8K seeds)")
print(f"Loaded {len(raw_completions)} raw completion rows")

## Config

All tunable parameters from `method.py`, unchanged from the original defaults. `N_SAMPLES` below reflects how many samples per `(prompt, model)` are actually present in the demo subset (5, vs. 20 in the full run) -- everything else is copied as-is from the original script.

In [ ]:
# Original method.py config (unchanged) -- N_SAMPLES is set to what's actually
# in the demo subset (5); the full run used N_SAMPLES=20.
MODELS = ["openai/gpt-4o-mini", "openai/gpt-4.1-mini", "openai/gpt-4.1-nano"]
N_SAMPLES = 5  # full run: 20
TEMPERATURE = 0.7
MAX_TOKENS = 400
TOP_LOGPROBS = 5
HARD_BUDGET_USD = 9.00
CONCURRENCY = 32
FIRST_K_TOKENS = 20

## Prompt design (from `build_dataset.py`)

For each GSM8K seed question, `build_dataset.py` generates 7 prompt variants: a bare-question control, plus `{relevant, filler}` content x `{short, medium, long}` length tiers, token-matched within each tier via the `cl100k_base` tokenizer. `matched_prompts.json` (loaded above as `prompts`) is the output of that script. We inspect the demo's 2 seeds x 7 variants = 14 prompts below to see the design directly, and re-verify the token-matching check `build_dataset.py` runs after generation (relevant vs. filler mean token count per tier).

In [ ]:
# Token-matching check, copied from build_dataset.py's main() -- verifies that
# 'relevant' and 'filler' prompts have near-identical token counts within each tier.
import statistics

df_prompts = pd.DataFrame(prompts)
for tier in ["short", "medium", "long"]:
    rel = [r["token_count"] for r in prompts if r["length_tier"] == tier and r["content_type"] == "relevant"]
    fil = [r["token_count"] for r in prompts if r["length_tier"] == tier and r["content_type"] == "filler"]
    logger.info(
        f"tier={tier} relevant mean_tok={statistics.mean(rel):.1f} filler mean_tok={statistics.mean(fil):.1f}"
    )

df_prompts[["prompt_id", "content_type", "length_tier", "token_count", "gold_answer"]]

## Answer extraction and entropy proxy (from `method.py`)

These are the exact helper functions `method.py` uses on each completion: `extract_numeric_answer` pulls the numeric answer out of the model's free text via a layered regex cascade, and `entropy_from_top_logprobs` / `locate_answer_token_index` compute the Shannon-entropy-in-nats proxy (a documented lower bound, since only the top-`k` logprobs are observed) at the first `FIRST_K_TOKENS` generated tokens and at the token where the numeric answer is emitted. The demo's `raw_completions` already carry the pre-computed `answer`, `mean_entropy_first_k`, and `answer_token_entropy` fields (produced by exactly this code during the real run), so we reproduce the functions here for reference/inspection rather than re-running them.

In [ ]:
ANSWER_PATTERNS = [
    re.compile(r"final answer\s*[:=]?\s*\$?(-?[\d,]*\.?\d+)", re.IGNORECASE),
    re.compile(r"\\boxed\{(-?[\d,]*\.?\d+)\}"),
    re.compile(r"\*\*\s*(-?[\d,]*\.?\d+)\s*\*\*"),
    re.compile(r"answer\s*[:=]?\s*\$?(-?[\d,]*\.?\d+)", re.IGNORECASE),
    re.compile(r"(-?[\d,]*\.?\d+)\s*\.?\s*$"),  # last resort: trailing number
]


def extract_numeric_answer(text: str):
    for pat in ANSWER_PATTERNS:
        m = pat.findall(text)
        if m:
            raw = m[-1].replace(",", "")
            try:
                return float(raw)
            except ValueError:
                continue
    return None


def entropy_from_top_logprobs(top_logprobs_list) -> float:
    """Shannon entropy (nats) of the visible top-k token distribution,
    renormalized over the observed mass. This is a LOWER BOUND on the true
    entropy since only the top-k token probabilities are observed."""
    lps = np.array([tl["logprob"] for tl in top_logprobs_list], dtype=np.float64)
    probs = np.exp(lps)
    s = probs.sum()
    if s <= 0:
        return 0.0
    probs = probs / s
    return float(scipy_entropy(probs))


def locate_answer_token_index(tokens: list, answer):
    """Find the token index whose text plausibly begins the numeric answer
    string, scanning from the end (answers are typically near the end)."""
    if answer is None:
        return None
    answer_str = ("%g" % answer).lstrip("-")
    for i in range(len(tokens) - 1, -1, -1):
        tok_txt = tokens[i]["token"].strip().replace(",", "")
        if tok_txt and (tok_txt in answer_str or answer_str.startswith(tok_txt)):
            return i
    return None


# Sanity check on one demo completion: re-run extraction/entropy and compare
# to the value logged during the real run.
sample_rec = raw_completions[0]
print("logged answer:", sample_rec["answer"], "| re-extracted:", extract_numeric_answer(sample_rec["raw_text"]))

## OpenRouter sampling (from `method.py`, reference only -- not executed)

The real run calls the OpenRouter chat-completions API `N_SAMPLES` times per `(prompt, model)`, with `logprobs=True` and a hard `$9.00` cost cap enforced by `RunningCost`, appending every raw completion to `outputs/raw_completions.jsonl` (resumable via `already_done_keys`). This is reproduced verbatim below for reference; it needs `OPENROUTER_API_KEY` and issues billed API calls, so this notebook does **not** execute it -- the demo instead loads the already-collected `raw_completions` from `mini_demo_data.json`, which is exactly what `load_raw_df()` would produce by reading `outputs/raw_completions.jsonl` after `run_all` finished.

In [ ]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"


class BudgetExceeded(Exception):
    pass


class RunningCost:
    def __init__(self, hard_budget: float):
        self.total = 0.0
        self.hard_budget = hard_budget
        self.lock = asyncio.Lock()

    async def add(self, cost: float):
        async with self.lock:
            self.total += cost
            if self.total > self.hard_budget:
                raise BudgetExceeded(f"cumulative cost {self.total:.4f} exceeded {self.hard_budget}")
            return self.total


RETRYABLE = (aiohttp.ClientError, asyncio.TimeoutError)


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type(RETRYABLE),
    reraise=True,
)
async def call_openrouter(session, model: str, prompt_text: str, api_key: str):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt_text}],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "logprobs": True,
        "top_logprobs": TOP_LOGPROBS,
    }
    async with session.post(
        OPENROUTER_URL,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json=payload,
        timeout=aiohttp.ClientTimeout(total=90),
    ) as resp:
        data = await resp.json()
        if resp.status == 429:
            raise aiohttp.ClientError(f"rate limited: {data}")
        if resp.status != 200:
            raise RuntimeError(f"HTTP {resp.status}: {json.dumps(data)[:500]}")
        if "choices" not in data:
            raise RuntimeError(f"malformed response, no choices: {json.dumps(data)[:500]}")
        return data


print("OpenRouter sampling functions defined (not called -- requires OPENROUTER_API_KEY and issues billed calls).")

## Aggregation: per-(prompt, model) variance, CV, accuracy, entropy

This is `aggregate_results` from `method.py`, unmodified. It groups the raw completions by `(prompt_id, model)` and, for every group with at least 2 valid (parseable) answers, computes the answer mean/sd/variance/CV, `frac_correct` against the gold GSM8K answer, and the mean of both logprob-entropy proxies. We run it directly on `raw_df`, the demo's `raw_completions` loaded into a DataFrame -- exactly the DataFrame `load_raw_df()` would produce from `outputs/raw_completions.jsonl`.

In [ ]:
def aggregate_results(raw_df: pd.DataFrame, n_samples_target: int) -> pd.DataFrame:
    results = []
    for (prompt_id, model), group in raw_df.groupby(["prompt_id", "model"]):
        valid = group.dropna(subset=["answer"])
        n_valid = len(valid)
        answers = valid["answer"].to_numpy(dtype=float)
        gold = group["gold_answer"].iloc[0]
        if n_valid >= 2:
            answer_mean = float(np.mean(answers))
            answer_sd = float(np.std(answers, ddof=1))
            answer_variance = float(np.var(answers, ddof=1))
            answer_cv = answer_sd / abs(answer_mean) if answer_mean != 0 else float("nan")
            frac_correct = float(np.mean(np.isclose(answers, gold, atol=1e-6)))
        else:
            answer_mean = float(answers[0]) if n_valid == 1 else float("nan")
            answer_sd = float("nan")
            answer_variance = float("nan")
            answer_cv = float("nan")
            frac_correct = float("nan")

        ent_fk = group["mean_entropy_first_k"].dropna()
        ent_ans = group["answer_token_entropy"].dropna()

        results.append(
            {
                "prompt_id": prompt_id,
                "model": model,
                "content_type": group["content_type"].iloc[0],
                "length_tier": group["length_tier"].iloc[0],
                "token_count": int(group["token_count"].iloc[0]),
                "gold_answer": gold,
                "n_samples_attempted": len(group),
                "n_valid_samples": n_valid,
                "pct_unparseable": 1 - n_valid / max(len(group), 1),
                "answer_mean": answer_mean,
                "answer_sd": answer_sd,
                "answer_variance": answer_variance,
                "answer_cv": answer_cv,
                "frac_correct": frac_correct,
                "mean_logprob_entropy_first_k": float(ent_fk.mean()) if len(ent_fk) else None,
                "mean_answer_token_entropy": float(ent_ans.mean()) if len(ent_ans) else None,
                "n_entropy_first_k_obs": int(len(ent_fk)),
                "n_answer_token_entropy_obs": int(len(ent_ans)),
                "low_n_flag": n_valid < 5,
            }
        )
    return pd.DataFrame(results)


raw_df = pd.DataFrame(raw_completions)
results_df = aggregate_results(raw_df, N_SAMPLES)
print(f"Aggregated {len(results_df)} (prompt, model) rows")
results_df.head(10)

## Summary statistics and the baseline comparison

`build_summary_stats` reports run-level stats (cost, logprob coverage, per-`content_type` x `length_tier` group means). `build_baseline_comparison` is the design's built-in baseline: the bare-question control vs. filler-padded vs. relevant-elaboration prompts at each length tier -- the core comparison behind the headline finding (both content types raise answer variance over the bare control, and the effect is non-monotonic in length). Both functions are copied unmodified from `method.py`.

In [ ]:
def build_summary_stats(results_df: pd.DataFrame, raw_df: pd.DataFrame, cost_tracker: RunningCost, models: list, budget_stopped: bool) -> dict:
    models_with_logprobs = sorted(raw_df.loc[raw_df["has_logprobs"], "model"].unique().tolist())
    models_without_logprobs = sorted(set(models) - set(models_with_logprobs))

    def group_mean(col):
        sub = results_df.dropna(subset=[col])
        if sub.empty:
            return {}
        g = sub.groupby(["content_type", "length_tier"])[col].mean()
        return {f"{a}|{b}": float(v) for (a, b), v in g.items()}

    return {
        "n_prompts": int(results_df["prompt_id"].nunique()),
        "n_models": len(models),
        "models_used": models,
        "n_total_calls_attempted": int(len(raw_df)) if not raw_df.empty else 0,
        "n_total_calls_succeeded": int(raw_df["answer"].notna().sum()) if not raw_df.empty else 0,
        "total_cost_usd": float(cost_tracker.total),
        "budget_stopped_early": bool(budget_stopped),
        "mean_cv_by_content_type_length_tier": group_mean("answer_cv"),
        "mean_entropy_first_k_by_content_type_length_tier": group_mean("mean_logprob_entropy_first_k"),
        "mean_answer_token_entropy_by_content_type_length_tier": group_mean("mean_answer_token_entropy"),
        "mean_frac_correct_by_content_type_length_tier": group_mean("frac_correct"),
        "pct_rows_low_n": float(results_df["low_n_flag"].mean()) if len(results_df) else None,
        "pct_rows_missing_logprobs": float(results_df["mean_logprob_entropy_first_k"].isna().mean()) if len(results_df) else None,
        "models_with_logprob_support": models_with_logprobs,
        "models_with_no_logprob_support": models_without_logprobs,
    }


def build_baseline_comparison(results_df: pd.DataFrame) -> dict:
    """Baseline comparison built into the design: bare-question control
    (no added content) vs the length-tiered relevant/filler variants, and
    filler-vs-relevant at matched length (content-effect isolation)."""
    out = {}
    bare = results_df[results_df["length_tier"] == "bare"]
    out["bare_control_mean_cv"] = float(bare["answer_cv"].dropna().mean()) if len(bare) else None
    out["bare_control_mean_frac_correct"] = float(bare["frac_correct"].dropna().mean()) if len(bare) else None
    for tier in ["short", "medium", "long"]:
        for ct in ["relevant", "filler"]:
            sub = results_df[(results_df["length_tier"] == tier) & (results_df["content_type"] == ct)]
            out[f"{ct}_{tier}_mean_cv"] = float(sub["answer_cv"].dropna().mean()) if len(sub) else None
            out[f"{ct}_{tier}_mean_frac_correct"] = float(sub["frac_correct"].dropna().mean()) if len(sub) else None
    return out


# Build a RunningCost from the demo's logged per-call costs (same field run_all's
# cost_tracker accumulates from) so build_summary_stats runs unmodified.
cost_tracker = RunningCost(HARD_BUDGET_USD)
cost_tracker.total = float(raw_df["cost"].sum())
budget_stopped = False

summary_stats = build_summary_stats(results_df, raw_df, cost_tracker, MODELS, budget_stopped)
baseline_comparison = build_baseline_comparison(results_df)
print(json.dumps(summary_stats, indent=2))
print(json.dumps(baseline_comparison, indent=2))

## Results

A readable table of mean answer CV and accuracy per `content_type` x `length_tier` cell (on this small demo subset), plus a bar chart of CV vs. length tier for `relevant` vs. `filler` content against the bare-control baseline -- the same shape of comparison the full run's headline result is built from. Because the demo only has 2 seeds x 5 samples, these numbers are noisy estimates, not a replication of the full-scale finding.

In [ ]:
group_table = (
    results_df.groupby(["content_type", "length_tier"])
    .agg(mean_cv=("answer_cv", "mean"), mean_frac_correct=("frac_correct", "mean"), n_rows=("prompt_id", "count"))
    .reset_index()
)
print("Mean answer CV / accuracy by content_type x length_tier (demo subset):")
print(group_table.to_string(index=False))

print(f"\
Total demo cost so far: ${summary_stats['total_cost_usd']:.4f} (full run: $2.0653, 6720 calls)")

tiers = ["short", "medium", "long"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bare_cv = baseline_comparison["bare_control_mean_cv"]
for ct, marker in [("relevant", "o"), ("filler", "s")]:
    cvs = [baseline_comparison[f"{ct}_{t}_mean_cv"] for t in tiers]
    axes[0].plot(tiers, cvs, marker=marker, label=ct)
if bare_cv is not None:
    axes[0].axhline(bare_cv, color="gray", linestyle="--", label="bare control")
axes[0].set_title("Answer CV vs. length tier")
axes[0].set_xlabel("length tier")
axes[0].set_ylabel("mean answer CV")
axes[0].legend()

for ct, marker in [("relevant", "o"), ("filler", "s")]:
    accs = [baseline_comparison[f"{ct}_{t}_mean_frac_correct"] for t in tiers]
    axes[1].plot(tiers, accs, marker=marker, label=ct)
bare_acc = baseline_comparison["bare_control_mean_frac_correct"]
if bare_acc is not None:
    axes[1].axhline(bare_acc, color="gray", linestyle="--", label="bare control")
axes[1].set_title("Accuracy vs. length tier")
axes[1].set_xlabel("length tier")
axes[1].set_ylabel("mean frac_correct")
axes[1].legend()

plt.tight_layout()
plt.show()